# 06 — Black-Litterman con Views de Momentum

Este notebook implementa paso a paso el modelo **Black-Litterman (1990)** aplicado al universo F5 del proyecto FinPUC, usando la señal de **momentum 12-1 meses** para generar automáticamente la matriz de views $P$ y el vector $Q$.

## Estructura
| Paso | Contenido |
|------|-----------|
| 1 | Carga de artefactos pre-computados |
| 2 | Prior de equilibrio de mercado $\pi$ |
| 3 | Señal de momentum y construcción de $P$, $Q$, $\Omega$ |
| 4 | Posterior bayesiano $\mu_{BL}$ |
| 5 | Comparación $\mu_{BL}$ vs $\mu_{histórico}$ |
| 6 | Optimización Markowitz con $\mu_{BL}$ |
| 7 | Comparación de portafolios BL vs Markowitz clásico |
| 8 | Análisis de sensibilidad ($\tau$, $\Omega$) |

**Prerequisitos:** Ejecutar `01_Preparar_Datos.ipynb` y `02_Modelo_Markowitz.ipynb` para generar los artefactos en `outputs/`.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Asegurar que el paquete solver sea encontrable
ROOT = Path('.').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from solver.black_litterman_views import (
    compute_bl_posterior,
    generate_momentum_views,
    load_artifacts,
    markowitz_with_bl,
    _compute_momentum_signal,
    MOMENTUM_WINDOW_DAYS,
    SKIP_DAYS,
    DEFAULT_TOP_K,
    DEFAULT_BOTTOM_K,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Módulos cargados correctamente.')

---
## Paso 1 — Carga de artefactos pre-computados

Se cargan los parámetros estimados por `02_Modelo_Markowitz.ipynb`:
- `mu` — retornos esperados anualizados (media semanal × 52)
- `sigma` — matriz de covarianza anualizada con shrinkage diagonal 10%
- `daily_returns` — retornos diarios logarítmicos del universo F5

In [ ]:
arts = load_artifacts()

mu: pd.Series         = arts['mu']            # (N,)
sigma: pd.DataFrame   = arts['sigma']          # (N, N)
daily_df: pd.DataFrame = arts['daily_returns'] # (T, N)

tickers   = mu.index.tolist()
mu_arr    = mu.values.astype(float)
sigma_arr = sigma.loc[tickers, tickers].values.astype(float)
daily_arr = daily_df[tickers].values.astype(float)
N         = len(tickers)

print(f'Universo F5: {N} activos')
print(f'Retornos diarios: {daily_arr.shape} | rango {daily_df.index.min().date()} – {daily_df.index.max().date()}')
print(f'mu  — min: {mu_arr.min():.2%}  media: {mu_arr.mean():.2%}  max: {mu_arr.max():.2%}')
print(f'sigma — shape: {sigma_arr.shape}')

---
## Paso 2 — Prior de equilibrio del mercado $\pi$

El modelo BL parte de la hipótesis de que el mercado está en **equilibrio CAPM** antes de incorporar views. Los retornos implícitos de equilibrio son:

$$\pi = r_f + \lambda \cdot \Sigma \, w_{mkt}$$

donde $w_{mkt}$ son los pesos por capitalización bursátil del universo F5 y $\lambda$ es la aversión al riesgo de mercado.

> **Nota:** Los artefactos `.pkl` no contienen market caps, por lo que aquí se usa $w_{mkt}$ equiponderado como proxy. En el webapp, los market caps reales se inyectan desde la base de datos SQLite.

In [ ]:
RISK_FREE    = 0.05    # tasa libre de riesgo anual
LAMBDA_RISK  = 2.5     # aversión al riesgo del mercado
TAU          = 0.05    # escala de incertidumbre sobre la covarianza

# Pesos de mercado (equiponderados como proxy)
market_caps    = np.ones(N, dtype=float)
market_weights = market_caps / market_caps.sum()

# Prior de equilibrio
pi = RISK_FREE + LAMBDA_RISK * (sigma_arr @ market_weights)

pi_series = pd.Series(pi, index=tickers)
print('Prior π — estadísticas:')
print(pi_series.describe().apply(lambda x: f'{x:.2%}'))

# Visualización
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(pi_series, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(pi_series.mean(), color='crimson', lw=2, label=f'Media = {pi_series.mean():.1%}')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('Distribución de la prior π (equilibrio de mercado)')
ax.set_xlabel('Retorno anualizado implícito')
ax.legend()
plt.tight_layout()
plt.show()

---
## Paso 3 — Señal de momentum y construcción de $P$, $Q$, $\Omega$

### 3.1 Señal de momentum 12-1

La señal mide el retorno acumulado de los últimos 12 meses **excluyendo el último mes** para evitar el efecto *reversal* de corto plazo:

$$\text{signal}_i = \prod_{t \in [T-273,\, T-21]} (1 + r_{t,i}) - 1$$

### 3.2 Construcción de views
- **Ganadores** (top 20 por momentum): $Q_k = \mu_{hist,k} \times 1.10$ — se espera que el momentum continúe.
- **Perdedores** (bottom 20 por momentum): $Q_k = \mu_{hist,k} \times 0.50$ — se espera deterioro respecto al histórico.

### 3.3 Calibración de $\Omega$ (He & Litterman 1999)

$$\Omega_{kk} = \tau \cdot P_k \Sigma P_k^\top$$

Esta calibración proporcional asegura que la incertidumbre de cada view sea coherente con la varianza del portafolio de esa view.

In [ ]:
TOP_K    = 20
BOTTOM_K = 20

# Señal de momentum raw
signal_raw = _compute_momentum_signal(daily_arr, MOMENTUM_WINDOW_DAYS, SKIP_DAYS)
signal_s   = pd.Series(signal_raw, index=tickers).sort_values(ascending=False)

print(f'Momentum 12-1 — Top 10 ganadores:')
print(signal_s.head(10).apply(lambda x: f'{x:.1%}').to_string())
print(f'\nMomentum 12-1 — Top 10 perdedores:')
print(signal_s.tail(10).apply(lambda x: f'{x:.1%}').to_string())

In [ ]:
# Generar views P, Q, Omega
P, Q, Omega, views_used = generate_momentum_views(
    tickers=tickers,
    daily_returns=daily_arr,
    mu_historical=mu_arr,
    ann_cov=sigma_arr,
    tau=TAU,
    top_k=TOP_K,
    bottom_k=BOTTOM_K,
)

K = len(Q)
print(f'P shape: {P.shape}   Q shape: {Q.shape}   Omega shape: {Omega.shape}')
print(f'Total de views: {K} ({TOP_K} ganadores + {BOTTOM_K} perdedores)')

# Tabla de views
views_df = pd.DataFrame(views_used)
print('\nPrimeras 5 views:')
display(views_df.head())
print('\nÚltimas 5 views:')
display(views_df.tail())

In [ ]:
# Visualización: distribución del momentum y views seleccionadas
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma del momentum
ax = axes[0]
ax.hist(signal_s, bins=50, color='slategray', edgecolor='white', alpha=0.7)
for v in views_df[views_df.tipo == 'ganador']['momentum_12_1_pct'] / 100:
    ax.axvline(v, color='seagreen', lw=0.5, alpha=0.6)
for v in views_df[views_df.tipo == 'perdedor']['momentum_12_1_pct'] / 100:
    ax.axvline(v, color='crimson', lw=0.5, alpha=0.6)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('Señal de momentum 12-1 (verde=ganadores, rojo=perdedores)')
ax.set_xlabel('Retorno acumulado 12-1m')

# Q vs mu_hist para activos con view
ax = axes[1]
colors = ['seagreen' if t == 'ganador' else 'crimson' for t in views_df.tipo]
ax.scatter(
    views_df.mu_hist_pct / 100,
    views_df.view_return_pct / 100,
    c=colors, alpha=0.8, s=50, edgecolors='white', lw=0.5
)
lim_max = views_df[['mu_hist_pct', 'view_return_pct']].max().max() / 100 * 1.1
lim_min = views_df[['mu_hist_pct', 'view_return_pct']].min().min() / 100 * 1.1
ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', lw=1, alpha=0.4, label='Q = μ_hist')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('μ_histórico')
ax.set_ylabel('Q (view asignada)')
ax.set_title('Views Q vs μ_histórico por tipo')
ax.legend()

plt.tight_layout()
plt.show()

---
## Paso 4 — Posterior bayesiano $\mu_{BL}$

La fórmula BL combina la prior $\pi$ con las views $(P, Q, \Omega)$:

$$M = \left[(\tau\Sigma)^{-1} + P^\top \Omega^{-1} P\right]^{-1}$$
$$\mu_{BL} = M \left[(\tau\Sigma)^{-1}\pi + P^\top \Omega^{-1} Q\right]$$

El resultado es un vector de retornos esperados que **pondera** la opinión del mercado con las views de momentum.

In [ ]:
mu_bl, pi_calc, info = compute_bl_posterior(
    tickers=tickers,
    ann_cov=sigma_arr,
    market_caps=market_caps,
    p_matrix=P,
    q_vector=Q,
    omega=Omega,
    risk_free=RISK_FREE,
    lambda_risk=LAMBDA_RISK,
    tau=TAU,
)

mu_bl_s = pd.Series(mu_bl, index=tickers)

print('=== Diagnóstico BL ===')
for k, v in info.items():
    if k != 'views_used':
        print(f'  {k}: {v}')

print('\n=== μ_BL — Estadísticas ===')
print(mu_bl_s.describe().apply(lambda x: f'{x:.2%}'))

---
## Paso 5 — Comparación $\mu_{BL}$ vs $\mu_{histórico}$

Se visualiza el impacto de las views: los activos con views positivas (ganadores de momentum) reciben un $\mu_{BL}$ superior a su $\mu_{hist}$; los perdedores, inferior.

In [ ]:
delta = mu_bl_s - mu
winner_tickers = [v['ticker'] for v in views_used if v['tipo'] == 'ganador']
loser_tickers  = [v['ticker'] for v in views_used if v['tipo'] == 'perdedor']
neutral_tickers = [t for t in tickers if t not in winner_tickers + loser_tickers]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter μ_BL vs μ_hist para todo el universo
ax = axes[0]
ax.scatter(mu[neutral_tickers], mu_bl_s[neutral_tickers],
           c='slategray', s=15, alpha=0.5, label='Sin view')
ax.scatter(mu[winner_tickers], mu_bl_s[winner_tickers],
           c='seagreen', s=40, alpha=0.8, label='Ganadores', zorder=3)
ax.scatter(mu[loser_tickers], mu_bl_s[loser_tickers],
           c='crimson', s=40, alpha=0.8, label='Perdedores', zorder=3)
lim = max(abs(mu.max()), abs(mu_bl_s.max())) * 1.05
ax.plot([-lim, lim], [-lim, lim], 'k--', lw=1, alpha=0.4)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('μ_histórico')
ax.set_ylabel('μ_BL')
ax.set_title('μ_BL vs μ_histórico (universo completo)')
ax.legend()

# Histograma de deltas
ax = axes[1]
ax.hist(delta[neutral_tickers], bins=40, color='slategray', alpha=0.6, label='Sin view')
ax.hist(delta[winner_tickers], bins=10, color='seagreen', alpha=0.7, label='Ganadores')
ax.hist(delta[loser_tickers], bins=10, color='crimson', alpha=0.7, label='Perdedores')
ax.axvline(0, color='black', lw=1)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('Δ = μ_BL − μ_hist')
ax.set_title('Impacto de las views sobre μ')
ax.legend()

plt.tight_layout()
plt.show()

print(f'Δ promedio ganadores:  {delta[winner_tickers].mean():.2%}')
print(f'Δ promedio perdedores: {delta[loser_tickers].mean():.2%}')
print(f'Δ promedio neutros:    {delta[neutral_tickers].mean():.2%}')

---
## Paso 6 — Optimización Markowitz con $\mu_{BL}$

Se optimiza el portafolio sustituyendo $\mu_{hist}$ por $\mu_{BL}$ en la función objetivo:

$$\max_w \; w^\top \mu_{BL} - \tfrac{1}{2} \gamma \, w^\top \Sigma w
\quad \text{s.t.} \sum w_i = 1,\; 0 \le w_i \le w_{max}$$

Se compara contra el portafolio Markowitz con $\mu_{hist}$ para el perfil Neutro (γ=18, w_max=10%).

In [ ]:
from markowitz_pipeline import optimize_markowitz, PROFILE_CONFIGS

profile = PROFILE_CONFIGS['Neutro']
GAMMA      = profile.gamma        # 18.0
MAX_WEIGHT = profile.max_weight   # 0.10
TOP_N      = 80                   # sub-universo del perfil Neutro

# Markowitz con μ_BL
w_bl, tickers_bl = markowitz_with_bl(
    mu_bl=mu_bl,
    ann_cov=sigma_arr,
    tickers=tickers,
    gamma=GAMMA,
    max_weight=MAX_WEIGHT,
    top_n=TOP_N,
)

# Markowitz clásico (μ_hist) — mismo sub-universo top-N por mu
top_n_idx   = np.argsort(mu_arr)[::-1][:TOP_N]
tickers_mk  = [tickers[i] for i in top_n_idx]
mu_sub      = mu_arr[top_n_idx]
sigma_sub   = sigma_arr[np.ix_(top_n_idx, top_n_idx)]
w_mk_arr, _ = optimize_markowitz(
    mu=pd.Series(mu_sub, index=tickers_mk),
    cov=pd.DataFrame(sigma_sub, index=tickers_mk, columns=tickers_mk),
    gamma=GAMMA,
    max_weight=MAX_WEIGHT,
)

# Métricas de portafolio
def portfolio_metrics(weights, tickers_sel):
    idx  = [tickers.index(t) for t in tickers_sel]
    mu_p = float(weights @ mu_arr[idx])
    sig_p = float(np.sqrt(weights @ sigma_arr[np.ix_(idx, idx)] @ weights))
    return {'retorno': mu_p, 'volatilidad': sig_p, 'sharpe': mu_p / sig_p}

m_bl = portfolio_metrics(w_bl, tickers_bl)
m_mk = portfolio_metrics(w_mk_arr, tickers_mk)

comp = pd.DataFrame([m_bl, m_mk], index=['BL (momentum)', 'Markowitz clásico'])
comp[['retorno', 'volatilidad']] = comp[['retorno', 'volatilidad']].applymap(lambda x: f'{x:.2%}')
comp['sharpe'] = comp['sharpe'].apply(lambda x: f'{x:.3f}')
print('=== Comparación de portafolios ===')
display(comp)

In [ ]:
# Top 10 holdings de cada portafolio
def top_holdings(weights, tickers_sel, n=10):
    s = pd.Series(weights, index=tickers_sel).sort_values(ascending=False)
    return s[s > 1e-4].head(n)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, w_arr, tick, label, color in [
    (axes[0], w_bl, tickers_bl, 'BL (momentum)', 'steelblue'),
    (axes[1], w_mk_arr, tickers_mk, 'Markowitz clásico', 'darkorange'),
]:
    h = top_holdings(w_arr, tick)
    ax.barh(h.index[::-1], h.values[::-1], color=color, alpha=0.85)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_title(f'Top holdings — {label}')
    ax.set_xlabel('Peso')

plt.tight_layout()
plt.show()

---
## Paso 7 — Frontera eficiente: BL vs Markowitz clásico

Se traza la frontera eficiente de ambos portafolios variando $\gamma$ para visualizar el desplazamiento que producen las views de momentum.

In [ ]:
gammas = np.logspace(0.3, 2.5, 25)
frontier_bl, frontier_mk = [], []

for g in gammas:
    w, t = markowitz_with_bl(mu_bl, sigma_arr, tickers, gamma=g, max_weight=MAX_WEIGHT, top_n=TOP_N)
    m = portfolio_metrics(w, t)
    frontier_bl.append((m['volatilidad'], m['retorno']))

    idx_n = np.argsort(mu_arr)[::-1][:TOP_N]
    t_n = [tickers[i] for i in idx_n]
    w_n, _ = optimize_markowitz(
        mu=pd.Series(mu_arr[idx_n], index=t_n),
        cov=pd.DataFrame(sigma_arr[np.ix_(idx_n, idx_n)], index=t_n, columns=t_n),
        gamma=g, max_weight=MAX_WEIGHT,
    )
    m_n = portfolio_metrics(w_n, t_n)
    frontier_mk.append((m_n['volatilidad'], m_n['retorno']))

f_bl = pd.DataFrame(frontier_bl, columns=['vol', 'ret'])
f_mk = pd.DataFrame(frontier_mk, columns=['vol', 'ret'])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(f_bl.vol, f_bl.ret, 'o-', color='steelblue', lw=2, ms=4, label='BL (momentum)')
ax.plot(f_mk.vol, f_mk.ret, 's-', color='darkorange', lw=2, ms=4, label='Markowitz clásico')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('Volatilidad anualizada')
ax.set_ylabel('Retorno anualizado')
ax.set_title('Frontera eficiente: BL vs Markowitz clásico (perfil Neutro, γ variable)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Paso 8 — Análisis de sensibilidad

### 8.1 Sensibilidad a $\tau$ (peso de la prior vs. views)

$\tau$ controla cuánto domina la prior de mercado vs. las views:
- $\tau \to 0$: $\mu_{BL} \approx \pi$ (la prior domina, views ignoradas)
- $\tau \to \infty$: $\mu_{BL} \approx$ solución dominada por views

In [ ]:
taus   = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
labels = [str(t) for t in taus]
sharpe_bl_tau = []

for t in taus:
    _, _, om_t, _ = generate_momentum_views(
        tickers=tickers, daily_returns=daily_arr, mu_historical=mu_arr,
        ann_cov=sigma_arr, tau=t, top_k=TOP_K, bottom_k=BOTTOM_K,
    )
    mu_t, _, _ = compute_bl_posterior(
        tickers=tickers, ann_cov=sigma_arr, market_caps=market_caps,
        p_matrix=P, q_vector=Q, omega=om_t,
        risk_free=RISK_FREE, lambda_risk=LAMBDA_RISK, tau=t,
    )
    w_t, tk_t = markowitz_with_bl(mu_t, sigma_arr, tickers, gamma=GAMMA,
                                   max_weight=MAX_WEIGHT, top_n=TOP_N)
    sharpe_bl_tau.append(portfolio_metrics(w_t, tk_t)['sharpe'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(labels, sharpe_bl_tau, 'o-', color='steelblue', lw=2, ms=6)
ax.axhline(m_mk['sharpe'], color='darkorange', lw=1.5, ls='--', label=f'Markowitz clásico ({m_mk["sharpe"]:.3f})')
ax.set_xlabel('τ')
ax.set_ylabel('Sharpe ratio')
ax.set_title('Sensibilidad del Sharpe BL al parámetro τ')
ax.legend()
plt.tight_layout()
plt.show()

### 8.2 Sensibilidad a $\Omega$ (confianza en las views)

Se reemplaza la calibración He-Litterman por un $\Omega$ plano con distintos valores.

In [ ]:
omegas = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50]
sharpe_bl_omega = []

for od in omegas:
    om_flat = np.eye(K, dtype=float) * od
    mu_o, _, _ = compute_bl_posterior(
        tickers=tickers, ann_cov=sigma_arr, market_caps=market_caps,
        p_matrix=P, q_vector=Q, omega=om_flat,
        risk_free=RISK_FREE, lambda_risk=LAMBDA_RISK, tau=TAU,
    )
    w_o, tk_o = markowitz_with_bl(mu_o, sigma_arr, tickers, gamma=GAMMA,
                                   max_weight=MAX_WEIGHT, top_n=TOP_N)
    sharpe_bl_omega.append(portfolio_metrics(w_o, tk_o)['sharpe'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([str(o) for o in omegas], sharpe_bl_omega, 's-', color='mediumpurple', lw=2, ms=6)
ax.axhline(m_mk['sharpe'], color='darkorange', lw=1.5, ls='--', label=f'Markowitz clásico ({m_mk["sharpe"]:.3f})')
ax.set_xlabel('Ω diagonal (plano)')
ax.set_ylabel('Sharpe ratio')
ax.set_title('Sensibilidad del Sharpe BL al parámetro Ω')
ax.legend()
plt.tight_layout()
plt.show()

---
## Resumen

| Componente | Valor / Descripción |
|------------|---------------------|
| Universo F5 | 638 activos |
| Views generadas | 40 (20 ganadores + 20 perdedores) |
| Señal | Momentum 12-1 meses (daily_returns) |
| Calibración Q | μ_hist × {1.10, 0.50} para ganadores/perdedores |
| Calibración Ω | He & Litterman: τ × P Σ Pᵀ |
| Prior π | rf + λ Σ w_mkt (equilibrio CAPM) |
| Optimización final | Markowitz SLSQP con μ_BL, mismo γ y w_max que perfil |

El módulo `solver/black_litterman_views.py` exporta todas las funciones utilizadas aquí. La webapp en `app/routers/portfolio.py` las consume mediante `views_source = "auto_momentum"`.